# Chapter 38 — Residual Stress Measurement: The Stress That Stays Behind
### *Modern Strain Gage Technology* — Companion Notebook

Some of the most dangerous stresses in an engineered part are the ones that are already there before it ever carries a load.

*Companion notebook to Chapter 38 — Residual Stress Measurement: The Stress That Stays Behind.*

---

**Residual stresses** are the stresses that remain locked in a material after every external load has been removed. They arise from manufacturing — heat treatment, welding, shot peening, machining — and their *sign* matters as much as their magnitude. Compressive residual stress at a surface is protective: it must be overcome before an applied tensile cycle can open a fatigue crack, which is exactly why shot peening and roller burnishing are worth the trouble. Tensile residual stress is the opposite. It adds to any applied tensile load, shortens fatigue life, and — as the chapter's opening account of cracked reactor coolant piping shows — can drive stress corrosion cracking in welds that passed every conventional inspection.

The most practical way to measure surface residual stress with strain gages is the **ASTM E837 blind hole-drilling** method. A small flat-bottomed hole (nominally 2 mm across for the common Type A rosette) is drilled at the center of a specialized three-element rosette. As material is removed, the residual stress around the hole relaxes, and the strain change recorded by each rosette element is converted back into the original residual stress state through a set of dimensionless calibration coefficients tabulated in the standard. This notebook implements that data reduction for a representative set of relaxation strains and then draws the resulting stress state as a Mohr's circle.

**This notebook demonstrates:**
1. **ASTM E837 blind hole-drilling data reduction** — three relaxation strain readings → principal residual stresses and their orientation.
2. **Visualizing the result** — the recovered residual stress state on a Mohr's circle.

In [ ]:
# !pip install numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

---
## 1 — ASTM E837 Blind Hole-Drilling Data Reduction

The reduction does not solve for $\sigma_x$, $\sigma_y$, and $\tau_{xy}$ directly. Instead it first folds the three relaxation strains $\varepsilon_1, \varepsilon_2, \varepsilon_3$ (elements at 0°, 45°, 90°) into three *combination strains* that cleanly separate the isotropic part of the relief from the two shear parts — an isotropic combination $\varepsilon_p = (\varepsilon_1 + \varepsilon_3)/2$, a deviatoric combination $\varepsilon_q = (\varepsilon_1 - \varepsilon_3)/2$, and a shear combination $\varepsilon_t = (\varepsilon_1 - 2\varepsilon_2 + \varepsilon_3)/2$. These are Chapter 38's Eqs. 38.1–38.3.

Each combination strain is proportional to the matching combination of residual stress through a dimensionless calibration coefficient — $\bar a$ for the isotropic part, $\bar b$ for the two shear parts. In words: divide each combination strain by its coefficient (and by the elastic constants) to recover a combination stress, then reassemble those into the principal stresses. The isotropic coefficient carries an extra $(1+\nu)$ factor, which is why we define Poisson's ratio here; the chapter's simplified Eqs. 38.1–38.4 fold that factor into the constant $A$ — the two forms are related by $A = -(1+\nu)\bar a/2$ and $B = -\bar b$ — but a numerically faithful reduction keeps it explicit, with the standard's own positive coefficients and its minus signs. Combining the three (Eqs. 38.4–38.5) gives the principal residual stresses $\sigma_{\max}, \sigma_{\min}$ and the orientation $\beta$ of the major principal axis relative to element 1.

**A word on sign.** As the chapter stresses, the sign returned is meaningful only once a convention is stated. We adopt the standard physical one: drilling into a field of *tensile* residual stress lets the surface contract as the tension is relieved, so the relaxation strains come out *negative*. Reading it the other way, the all-negative relaxation strains used below (−180, −120, −40 µε) correspond to a **tensile** residual stress state — the kind welding leaves behind, and precisely the condition implicated in the stress corrosion cracking that opens the chapter. With this convention the reduction returns tensile stresses as positive.

In [ ]:
# --- Measured inputs: the three relaxation strains after full-depth drilling ---
# Elements 1, 2, 3 sit at 0 deg, 45 deg, 90 deg from the rosette reference axis.
EPS_1_UE = -180.0   # relaxation strain, element 1 (0 deg),  microstrain
EPS_2_UE = -120.0   # relaxation strain, element 2 (45 deg), microstrain
EPS_3_UE =  -40.0   # relaxation strain, element 3 (90 deg), microstrain

# --- Material properties and ASTM E837 calibration coefficients ---------------
E_MPA = 200.0e3     # Young's modulus, MPa (structural steel)
NU    = 0.29        # Poisson's ratio, dimensionless
A_BAR = 0.0750      # isotropic calibration coefficient (a-bar), dimensionless
B_BAR = 0.2950      # shear calibration coefficient (b-bar), dimensionless
#   a_bar and b_bar are geometry-dependent constants tabulated in ASTM E837 for
#   the chosen rosette, hole diameter, and normalized depth h/D.


def reduce_hole_drilling(eps1_ue, eps2_ue, eps3_ue, E, nu, a_bar, b_bar):
    """Reduce three relaxation strains to principal residual stresses (ASTM E837).

    Implements the uniform-stress calculation of ASTM E837 (Chapter 38,
    Eqs. 38.1-38.5). The three microstrain inputs are the relaxation strains
    from the rosette elements at 0, 45, and 90 degrees; ``a_bar`` and ``b_bar``
    are the dimensionless E837 calibration coefficients for the rosette, hole
    diameter, and depth.

    Sign convention (see the markdown above): negative relaxation strains
    correspond to *tensile* residual stress, so tensile stress is returned as
    positive. Stresses are returned in the same units as ``E``.

    Returns
    -------
    (sigma_max, sigma_min, tau_max, beta_deg)
        Principal residual stresses, maximum in-plane shear stress, and the
        angle of the major principal axis measured from element 1, in degrees.
    """
    e1, e2, e3 = eps1_ue * 1e-6, eps2_ue * 1e-6, eps3_ue * 1e-6

    # Combination strains (Eqs. 38.1-38.3): isotropic, deviatoric, shear.
    eps_p = (e1 + e3) / 2.0
    eps_q = (e1 - e3) / 2.0
    eps_t = (e1 - 2.0 * e2 + e3) / 2.0

    # Combination stresses, inverting E837's P = -(1+nu) a_bar p / E and
    # Q = -b_bar q / E, T = -b_bar t / E (the E837 coefficients are positive).
    # The isotropic term carries the (1 + nu) factor that the chapter's
    # simplified constant A absorbs: A = -(1 + nu) a_bar / 2, B = -b_bar.
    P = -E * eps_p / (a_bar * (1.0 + nu))  # mean in-plane stress, (sx + sy)/2
    Q = -E * eps_q / b_bar                 # deviatoric stress,     (sx - sy)/2
    T = -E * eps_t / b_bar                 # in-plane shear stress, tau_xy

    tau_max = np.hypot(Q, T)               # radius of Mohr's circle
    sigma_max = P + tau_max
    sigma_min = P - tau_max
    beta_deg = 0.5 * np.degrees(np.arctan2(eps_t, eps_q))   # Eq. 38.5
    return sigma_max, sigma_min, tau_max, beta_deg


sigma_max, sigma_min, tau_max, beta_deg = reduce_hole_drilling(
    EPS_1_UE, EPS_2_UE, EPS_3_UE, E_MPA, NU, A_BAR, B_BAR)

print("ASTM E837 Blind Hole-Drilling Results")
print(f"  Relaxation strains: e1={EPS_1_UE:.0f}, e2={EPS_2_UE:.0f}, e3={EPS_3_UE:.0f} microstrain")
print(f"  sigma_max = {sigma_max:+.1f} MPa   (major principal residual stress)")
print(f"  sigma_min = {sigma_min:+.1f} MPa   (minor principal residual stress)")
print(f"  tau_max   = {tau_max:.1f} MPa   (max in-plane shear stress)")
print(f"  beta      = {beta_deg:+.2f} deg   (major principal axis, from element 1)")
print()
print("Positive = tensile residual stress (adds to applied load; harmful for fatigue and SCC).")

---
## 2 — The Residual Stress State as a Mohr's Circle

A pair of principal stresses and an angle is exactly the information a Mohr's circle encodes, so it is the natural picture for a hole-drilling result. Building on the construction from Chapter 17, the circle is centered on the mean normal stress $P = (\sigma_{\max} + \sigma_{\min})/2$ and has radius equal to the maximum in-plane shear stress $\tau_{\max} = (\sigma_{\max} - \sigma_{\min})/2$. Where the circle crosses the horizontal axis marks the two principal stresses; that the whole circle sits to the right of $\sigma = 0$ is the visual signature of a fully tensile residual field — every direction on the surface is in tension, which is why a state like this is so effective at driving stress corrosion cracking.

In [ ]:
def plot_mohr_circle(sigma_max, sigma_min, tau_max, ax=None):
    """Draw the Mohr's circle for an in-plane principal stress state.

    The circle is centered at the mean normal stress and its radius is the
    maximum in-plane shear stress; the two principal stresses are the points
    where it crosses the sigma axis.
    """
    center = 0.5 * (sigma_max + sigma_min)
    phi = np.linspace(0.0, 2.0 * np.pi, 361)

    if ax is None:
        _, ax = plt.subplots(figsize=(5.2, 5.2))

    ax.plot(center + tau_max * np.cos(phi), tau_max * np.sin(phi),
            color='C0', lw=2, label="Mohr's circle")
    ax.scatter([sigma_max, sigma_min], [0.0, 0.0], color='C3', zorder=5)
    ax.annotate(rf'$\sigma_{{\max}}={sigma_max:.0f}$ MPa', (sigma_max, 0.0),
                textcoords='offset points', xytext=(6, 10))
    ax.annotate(rf'$\sigma_{{\min}}={sigma_min:.0f}$ MPa', (sigma_min, 0.0),
                textcoords='offset points', xytext=(-6, 10), ha='right')
    ax.annotate(rf'$\tau_{{\max}}={tau_max:.0f}$ MPa', (center, tau_max),
                textcoords='offset points', xytext=(6, 4))

    ax.axhline(0.0, color='k', lw=0.8)
    ax.axvline(0.0, color='0.6', lw=0.8, ls='--')   # sigma = 0 reference
    ax.set_xlabel(r'Normal stress $\sigma$ (MPa)')
    ax.set_ylabel(r'Shear stress $\tau$ (MPa)')
    ax.set_title("Mohr's circle of the measured residual stress state")
    ax.set_aspect('equal')
    ax.grid(alpha=0.3)
    ax.legend(loc='lower right')
    return ax


plot_mohr_circle(sigma_max, sigma_min, tau_max)
plt.tight_layout()
plt.show()

---
## Where to take this next

The reduction above collapses a full test to a single uniform-stress answer. A few directions turn it into a real analysis tool:

1. **Flip the sign to model shot peening.** Feed in *positive* relaxation strains (e.g. +180, +120, +40 µε) and watch the Mohr's circle slide left of $\sigma = 0$ — a compressive, fatigue-protective field, the intended result of shot peening or low-plasticity burnishing. Compare it directly against the tensile weld case here.
2. **Reduce a depth series with the integral method.** Real E837 tests record strains at twenty increasing depths (0.05 mm steps for a Type A rosette). Replace the single reading with a depth table, apply the depth-dependent coefficient matrix, and plot residual stress versus depth — then add Tikhonov regularization (Chapter 38's non-uniform method) and study how the smoothing parameter trades noise against resolution.
3. **Propagate measurement uncertainty.** Wrap `reduce_hole_drilling` in a Monte Carlo loop that perturbs each strain by a realistic ±3 µε and the hole eccentricity within ±0.04D, then report confidence bands on $\sigma_{\max}$, $\sigma_{\min}$, and $\beta$ — a concrete lead-in to the uncertainty methods of Part VIII.